In [2]:
import pandas as pd
import tgt_certs
import os
import csv
import requests
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Optional, Dict, Iterable

tgt_certs.setup_env()
tgt_certs.setup_requests()

In [3]:
df = pd.read_csv("tcins_with_titles.csv")

unique_tcin = df['tcin'].unique()
print(df.shape)
print("Unique number of tcin in evaluation data:", len(unique_tcin))

(8473041, 2)
Unique number of tcin in evaluation data: 8473041


In [5]:
IMAGE_GRAPHQL_URL = "https://api-internal.target.com/items/v4/graphql/compact/item"
IMAGE_GRAPHQL_PARAMS = {
    "selection": "AFIBAAAATGPQ-MH0j5GBgYMBEzA-ZsAKBMCKmdFET7ADAA==",
    "x-api-key": "376f7b28eded53449ccd5bfdc97668bdf4c1fa34",
}

BATCH_SIZE = 10_000

def _make_session(pool: int) -> requests.Session:
    s = requests.Session()
    s.mount("https://", HTTPAdapter(pool_connections=pool, pool_maxsize=pool))
    s.mount("http://",  HTTPAdapter(pool_connections=pool, pool_maxsize=pool))
    return s

def _fetch_single(tcin: str, session: requests.Session, timeout=(3.0, 5.0)) -> Dict[str, Optional[str]]:
    try:
        r = session.get(
            IMAGE_GRAPHQL_URL,
            params={"selection": IMAGE_GRAPHQL_PARAMS["selection"], "tcin": tcin},
            headers={"x-api-key": IMAGE_GRAPHQL_PARAMS["x-api-key"]},
            timeout=timeout,
            verify=True,
        )
        if r.status_code != 200:
            return {"tcin": tcin, "image_url": None}
        item = (r.json().get("data") or {}).get("item") or {}
        imgs = (item.get("digitalAssets") or {}).get("images") or {}
        base = (imgs.get("baseUrl") or "").lstrip("/")
        prim = (imgs.get("primaryImage") or "").lstrip("/")
        if not (base or prim):
            return {"tcin": tcin, "image_url": None}
        url = f"https://{base}{prim}" if not base.startswith("http") else f"{base}{prim}"
        return {"tcin": tcin, "image_url": url}
    except Exception:
        return {"tcin": tcin, "image_url": None}

def _chunks(xs: Iterable[str], n: int):
    buf = []
    for x in xs:
        buf.append(x)
        if len(buf) == n:
            yield buf
            buf = []
    if buf:
        yield buf

def fetch_and_save_to_csv(
    product_ids: List[str],
    output_csv: str,
    max_workers: int = 32,
    batch_size: int = BATCH_SIZE,
    flush_every: int = 1000,
):
    os.makedirs(os.path.dirname(output_csv) or ".", exist_ok=True)
    total = len(product_ids)

    with open(output_csv, "w", newline="", encoding="utf-8") as f, \
         _make_session(max_workers) as session, \
         ThreadPoolExecutor(max_workers=max_workers) as ex, \
         tqdm(total=total, desc="Fetching Image URLs", unit="item", dynamic_ncols=True) as pbar:

        writer = csv.writer(f)
        writer.writerow(["tcin", "image_url"])

        written = 0
        for batch in _chunks(product_ids, batch_size):

            for res in ex.map(lambda x: _fetch_single(x, session), batch, chunksize=1):
                writer.writerow([res["tcin"], res["image_url"]])
                written += 1
                pbar.update(1)
                if written % flush_every == 0:
                    f.flush()

            f.flush()


In [6]:
fetch_and_save_to_csv(
    product_ids=unique_tcin,
    output_csv="image_urls.csv",
    max_workers=64,
)

Fetching Image URLs: 100%|██████████| 8473041/8473041 [6:53:51<00:00, 341.22item/s]   


# Check results

In [3]:
image_url = pd.read_csv("image_urls.csv")

null_image = image_url["image_url"].isna().sum()

print("Number of products with null image_url:", null_image, "\nAccount for", f"{null_image / len(image_url) * 100:.2f} %")

Number of products with null image_url: 45 
Account for 0.00 %


In [4]:
dup_images = image_url[image_url["image_url"].duplicated(keep=False)]
dup_images.shape

(5582152, 2)

In [5]:
tcin_title = pd.read_csv("tcins_with_titles.csv")
dup_images_with_title = dup_images.merge(tcin_title, on="tcin", how="left")
dup_images_with_title.sort_values(by="tcin").head(20)

,tcin,image_url,p_title
3774372,523061,https://target.scene7.com/is/image/Target/GUES...,Chicago Women's Rink Roller Skates - White (5)
3774373,523062,https://target.scene7.com/is/image/Target/GUES...,Chicago Women's Rink Roller Skates - White (6)
3774374,523064,https://target.scene7.com/is/image/Target/GUES...,Chicago Women's Rink Roller Skates - White (8)
3774375,523065,https://target.scene7.com/is/image/Target/GUES...,Chicago Women's Rink Roller Skates - White (9)
3774376,523066,https://target.scene7.com/is/image/Target/GUES...,Chicago Women's Rink Roller Skates - White (10)
3776742,523096,https://target.scene7.com/is/image/Target/GUES...,Chicago Men's Rink Roller Skates - 9
3776743,523100,https://target.scene7.com/is/image/Target/GUES...,Chicago Men's Rink Roller Skates - 11
3774436,540072,https://target.scene7.com/is/image/Target/GUES...,Chicago Skates Men's Bullet Speed Skate - Size 4
3774437,540076,https://target.scene7.com/is/image/Target/GUES...,Chicago Skates Men's Bullet Speed Skate - Size 6
3784256,598498,https://target.scene7.com/is/image/Target/GUES...,Melissa & Doug Magnetic Chalkboard and Dry-Era...


# Dedup and merge with title

In [6]:
df_clip = image_url.merge(tcin_title, on="tcin", how="inner")

In [7]:
df_clip.head()

,tcin,image_url,p_title
0,1000000005,https://target.scene7.com/is/image/Target/GUES...,Mosser Glass 7 inch Bowl - White Milk Glass
1,1002445430,https://target.scene7.com/is/image/Target/GUES...,Gracie Mills Penny Southwest-Inspired 5 Piece ...
2,93910569,https://target.scene7.com/is/image/Target/GUES...,Gracie Mills Penny Southwest-Inspired 5 Piece ...
3,93910570,https://target.scene7.com/is/image/Target/GUES...,Gracie Mills Penny Southwest-Inspired 5 Piece ...
4,1000000016,https://target.scene7.com/is/image/Target/GUES...,Square Enix - SQUARE ENIX - Kingdom Hearts III...


In [8]:
def clean_clip_df(df_clip):

    df = df_clip.dropna(subset=["image_url"])
    df = df.dropna(subset=["p_title"])
    
    df_unique = df.groupby("image_url", group_keys=False).sample(1, random_state=42)
    return df_unique

In [9]:
df = clean_clip_df(df_clip)
print("Final cleaned dataframe shape:", df.shape)

Final cleaned dataframe shape: (3976142, 3)


In [14]:
df.to_csv("tcin_title_image_url.csv", index=False)